# 82일차 · RAG 검색·답변 평가

정답 조항이 두 번째 후보로 밀린 날과 답변에 붙인 인용이 틀린 날을 구별합니다. 같은 질문에 검색 결과와 답변을 각각 채점하세요.

정답 조항 ID로 Recall@K·MRR@K를 계산한다. 기본 검색과 재정렬을 비교하고, 답변 정확성과 인용 근거가 답을 뒷받침하는지를 따로 평가한다.

## 정답 조항을 몇 개 찾았나요?

In [ ]:
gold = {"A2", "B1"}
baseline = ["A1", "B1", "A2"]
candidate = ["A2", "B1", "A1"]
def scores(ranking, gold, k):
    top = ranking[:k]
    hits = len(set(top) & gold)
    first = next((i for i, doc in enumerate(top, 1) if doc in gold), None)
    return hits / len(gold), 1 / first if first else 0

print(scores(baseline, gold, 2))
print(scores(candidate, gold, 2))

(0.5, 0.5)
(1.0, 1.0)


기준선은 필요한 두 조항 가운데 B1만 찾았습니다. 첫 적중은 2위라 역순위도 0.5입니다. 여기서는 한 질문의 역순위를 구했습니다. 여러 질문의 역순위를 평균하면 MRR@2가 됩니다.

### 직접 해 보기

두 번째 질문은 정답이 C1인데 두 검색기 모두 놓쳤습니다. 두 질문의 평균 Recall@2와 MRR@2를 계산하세요. 이어 답변 90일이 맞지만 A1(30일)을 인용한 경우를 별도로 판정하세요.

In [ ]:
second = (["C2", "C3"], {"C1"})
clauses = {"A1": 30, "A2": 90}
response = {"days": 90, "citation": "A1"}
# 검색 설정별 평균 점수, 답변 일치, 인용 지지를 출력하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre style="white-space:pre-wrap;overflow:auto"><code>second = ([&quot;C2&quot;, &quot;C3&quot;], {&quot;C1&quot;})
clauses = {&quot;A1&quot;: 30, &quot;A2&quot;: 90}
response = {&quot;days&quot;: 90, &quot;citation&quot;: &quot;A1&quot;}
for ranking in [baseline, candidate]:
    results = [scores(ranking, gold, 2), scores(*second, 2)]
    print(tuple(sum(column) / len(results) for column in zip(*results)))
print(response[&quot;days&quot;] == 90, clauses.get(response[&quot;citation&quot;]) == response[&quot;days&quot;])</code></pre><pre style="white-space:pre-wrap;overflow:auto"><code>(0.25, 0.25)
(0.5, 0.5)
True False
</code></pre></details>

## 수업 서비스에 적용하기

수업용 LangSmith 계정에 인증한 뒤 langsmith 패키지를 준비하세요. 셀이 개발 데이터셋을 만들고 세 날짜의 기준 문항을 채웁니다. 아래 셀의 evaluate는 같은 데이터셋에서 두 검색 함수를 비교합니다. 수업의 Retriever와 재정렬기를 연결할 때에도 문서·질문·K를 고정하세요. 답변 값과 인용 지지는 별도 열로 기록하고, 답할 근거가 없는 문항은 보류 여부를 채점하세요.

In [ ]:
run_integration = False
if run_integration:
    from langsmith import Client
    def correct_clause(outputs, reference_outputs):
        return outputs["citation"] == reference_outputs["citation"]
    def baseline_search(inputs):
        return {"citation": "A1"}
    def candidate_search(inputs):
        day = inputs["as_of"]
        return {"citation": "A2" if day >= "2026-07-01" else "A1" if day >= "2026-06-01" else None}
    client = Client()
    dataset_name = "course-product-qa-dev"
    if not client.has_dataset(dataset_name=dataset_name):
        client.create_dataset(dataset_name=dataset_name)
    dataset = client.read_dataset(dataset_name=dataset_name)
    seen = {e.inputs["as_of"] for e in client.list_examples(dataset_id=dataset.id)}
    for day, citation in [("2026-05-31", None), ("2026-06-30", "A1"), ("2026-07-02", "A2")]:
        if day not in seen:
            client.create_example(inputs={"as_of": day}, outputs={"citation": citation}, dataset_id=dataset.id)
    for name, target in [("baseline", baseline_search), ("candidate", candidate_search)]:
        client.evaluate(target, data="course-product-qa-dev",
                        evaluators=[correct_clause], experiment_prefix=name)

### 제출 전 확인

정답 조항 ID로 Recall@K·MRR@K를 계산한다. 기본 검색과 재정렬을 비교하고, 답변 정확성과 인용 근거가 답을 뒷받침하는지를 따로 평가한다.

[공식 문서](https://docs.langchain.com/langsmith/evaluate-llm-application)